# Push ORDM ODCS contracts to Ontos

Publishes ODCS contracts from a **workspace folder** into a running **Ontos** deployment, using the same `push_contracts_to_ontos.py` that runs from the command line. This notebook only obtains a token and calls the tool's importable functions — no logic is duplicated.

**Layout.** This notebook lives in the repo's `runtime/examples/notebooks/` folder; the tool lives in the sibling `tools/` folder. The import cell derives the repo root from this notebook's path and puts `tools/` on `sys.path`, so nothing is hardcoded. The Contracts folder is a separate parameter, so contracts can live anywhere the cluster can read (a repo path, a Volume, etc.).

**Prerequisites**
- This notebook and `tools/push_contracts_to_ontos.py` are checked out together (e.g. a Git folder under `/Workspace/Repos/...`).
- A bearer token the Ontos app accepts. Ontos runs as a Databricks App behind OAuth; when it runs in **this** workspace, `token_source=current_user` obtains an OAuth token for you via the Databricks SDK. A **secret scope** is the alternative (e.g. for a service principal, or a different workspace).

**Known caveat (Ontos-side, not this notebook):** Ontos's importer currently preserves schema, columns, descriptions, and domain, but does **not** yet persist primary keys or object-level quality rules, and re-uploading creates a new contract rather than updating. See `docs/odcs-integration.md`.


## 1. Parameters


In [ ]:
# Re-run this cell after editing to (re)create the widgets.
dbutils.widgets.text('contracts_dir', '/Workspace/Repos/me/staging/contracts/odcs', 'Contracts folder (holds *.odcs.yaml)')
dbutils.widgets.text('app_url', '', 'Ontos app URL (https://<app>.databricksapps.com)')
dbutils.widgets.text('contracts', '', 'Contracts (comma-separated unit names; blank = all)')
dbutils.widgets.dropdown('token_source', 'current_user', ['current_user', 'secret'], 'Token source')
dbutils.widgets.text('secret_scope', 'ontos', 'Secret scope (when token_source=secret)')
dbutils.widgets.text('secret_key', 'push_token', 'Secret key (when token_source=secret)')
dbutils.widgets.dropdown('dry_run', 'true', ['true', 'false'], 'Dry run (preview only)')


In [ ]:
contracts_dir = dbutils.widgets.get('contracts_dir').rstrip('/')
app_url       = dbutils.widgets.get('app_url').rstrip('/')
contracts_arg = dbutils.widgets.get('contracts')
token_source  = dbutils.widgets.get('token_source')
secret_scope  = dbutils.widgets.get('secret_scope')
secret_key    = dbutils.widgets.get('secret_key')
dry_run       = dbutils.widgets.get('dry_run') == 'true'

assert app_url, 'Set the app_url widget to your Ontos app URL.'
assert contracts_dir, 'Set the contracts_dir widget to the folder holding *.odcs.yaml.'
# blank Contracts = push everything found under contracts_dir
include = [c.strip() for c in contracts_arg.split(',') if c.strip()] or None
print(f'contracts_dir={contracts_dir}\napp_url={app_url}\ncontracts={include or "ALL"}\ndry_run={dry_run}')


## 2. Import the tool from the sibling `tools/` folder

This notebook is in `runtime/examples/notebooks/`; the tool is in `tools/`. We derive the repo root from this notebook's own path (via the notebook context) and add `<repo>/tools` to `sys.path` — no hardcoded repo location.


In [ ]:
import os, sys

def _add_tools_to_syspath():
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    nb_path = ctx.notebookPath().get()          # e.g. /Repos/me/staging/notebooks/push_contracts_to_ontos
    repo = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(nb_path))))   # -> .../staging
    tools = os.path.join(repo, 'tools')
    added = []
    for cand in (tools, '/Workspace' + tools):   # workspace-files FUSE mount is under /Workspace
        if os.path.isdir(cand) and cand not in sys.path:
            sys.path.insert(0, cand); added.append(cand)
    return added

print('added to sys.path:', _add_tools_to_syspath() or '(already importable)')
from push_contracts_to_ontos import (
    DEFAULT_CONTRACTS_GLOB, UPLOAD_PATH,
    find_contracts, push_paths, build_ssl_context,
)
print('imported push_contracts_to_ontos OK')


## 3. Authenticate

**`current_user` (default):** exchanges the notebook's internal token for an audience-scoped
OAuth JWT via the workspace OIDC token-exchange endpoint, scoped to the Ontos app's OAuth
client ID. Works on all compute types (including serverless) when Ontos runs in **this**
workspace and you have CAN USE on the app.

**`secret`:** reads a token from a secret scope (`databricks secrets put-secret <scope> <key>`).
Use this for a service principal, a token scoped to a different workspace, or CI.


In [ ]:
import re as _re
import requests as _req
from databricks.sdk import WorkspaceClient

def get_token(source):
    if source == 'current_user':
        w = WorkspaceClient()

        # The Apps proxy requires an audience-scoped OAuth token.  On any
        # compute (including serverless) we obtain one via the OIDC token-
        # exchange endpoint, exchanging the notebook's internal token for a
        # JWT scoped to the Ontos app's OAuth client ID.
        # URL format: https://<app_name>-<workspace_id>.<cloud>.databricksapps.com
        subdomain = app_url.split('//')[1].split('.')[0]
        app_name = _re.sub(r'-[0-9]+$', '', subdomain)  # strip numeric workspace-id suffix
        app_client_id = w.apps.get(app_name).oauth2_app_client_id

        notebook_token = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook().getContext().apiToken().get()
        )
        resp = _req.post(
            f'{w.config.host}/oidc/v1/token',
            data={
                'grant_type': 'urn:ietf:params:oauth:grant-type:token-exchange',
                'subject_token': notebook_token,
                'subject_token_type': 'urn:databricks:params:oauth:token-type:personal-access-token',
                'requested_token_type': 'urn:ietf:params:oauth:token-type:access_token',
                'scope': 'all-apis',
                'audience': app_client_id,
            },
        )
        if resp.status_code != 200:
            raise SystemExit(f'Token exchange failed ({resp.status_code}): {resp.text[:200]}')
        return resp.json()['access_token']
    if source == 'secret':
        tok = dbutils.secrets.get(scope=secret_scope, key=secret_key)
        if not tok:
            raise SystemExit(f'secret {secret_scope}/{secret_key} is empty')
        return tok
    raise SystemExit(f'unknown token_source: {source}')

token = None if dry_run else get_token(token_source)
print('token acquired' if token else 'dry-run: no token needed')

## 4. Select contracts

Reuses the tool's `find_contracts`, globbing `*.odcs.yaml` directly under the Contracts folder.
A blank **Contracts** widget pushes everything found; naming units restricts to those.


In [ ]:
# contracts_dir points straight at the folder of *.odcs.yaml, so glob that folder directly
# (rather than DEFAULT_CONTRACTS_GLOB, which is repo-root relative: contracts/odcs/*.odcs.yaml).
paths = find_contracts(contracts_dir, '*.odcs.yaml', include)
if not paths:
    raise SystemExit(f'no *.odcs.yaml found under {contracts_dir}')

print(f'{len(paths)} contract(s) selected for {app_url}{UPLOAD_PATH}:')
for p in paths:
    print('  -', p.split('/')[-1])


## 5. Push (or preview)


In [ ]:
if dry_run:
    print(f'[dry-run] would push {len(paths)} contract(s) — nothing sent.')
    results = [(p.split('/')[-1].replace('.odcs.yaml',''), None, 'dry-run') for p in paths]
else:
    results = push_paths(app_url, token, paths, ssl_ctx=build_ssl_context())

# Render a table (Databricks display); falls back to print off-platform.
rows = [{'unit': u, 'ok': ok, 'message': msg} for (u, ok, msg) in results]
try:
    display(spark.createDataFrame(rows))
except Exception:
    for r in rows:
        print(r)

failures = sum(1 for (_, ok, _) in results if ok is False)
print(f'done: {len(results) - failures} ok, {failures} failed'      + (' (dry-run)' if dry_run else ''))
if failures:
    raise SystemExit(f'{failures} contract(s) failed to push')


---
Runs the identical logic as `python tools/push_contracts_to_ontos.py --app-url <URL> --profile <P> --all`, but reads contracts from the Contracts-folder parameter and resolves the token from the current user's SDK OAuth or a secret scope. Everything after the token — selection, TLS, upload — is the shared tool code in `tools/`.
